# Probability Foundations — Part 4
## Statistical Inference and Likelihood Inference

**Coverage:** Chapter 5 — *Statistical Inference* and Chapter 6 — *Likelihood Inference*

This notebook summarizes the main learning process from the session: the textbook material, the questions that came up while reading, the mathematical derivations we worked through, and the conceptual corrections that became important.

The goal is **not** to reproduce every subsection of the textbook. In line with the updated learning plan, the emphasis is on ideas that matter most for later machine learning:

- statistical models and parameters;
- sampling and the population/sample distinction;
- likelihood and sufficient statistics;
- maximum likelihood estimation (MLE);
- estimator quality: bias, variance, MSE, standard error, consistency;
- confidence intervals and hypothesis testing;
- P-values and z-tests;
- distribution-free methods, especially bootstrapping.

More specialized or lower-priority details are kept brief.

## 1. The conceptual shift: probability $\rightarrow$ statistical inference

Earlier probability chapters mostly used the direction

$$
\text{known probability model} \longrightarrow \text{reason about unknown outcomes}.
$$

Statistical inference reverses the problem:

$$
\boxed{
\text{observed data}
\longrightarrow
\text{reason about the unknown probability model}
}
$$

This is the key transition in Chapter 5.

A **statistical model** is a family of candidate probability distributions,

$$
\mathcal{P} = \{P_\theta:\theta\in\Omega\},
$$

where

- $\theta$ is the **parameter**;
- $\Omega$ is the **parameter space**;
- each value of $\theta$ identifies one candidate distribution $P_\theta$.

The observed data tell us something about which member of this family is plausible.

### 1.1 What exactly is $\theta$?

One of the main conceptual questions in this session was what the parameter $\theta$ really represents.

A useful interpretation is:

$$
\boxed{
\theta = \text{the unknown quantity (or vector of quantities) that indexes the model family.}
}
$$

It is not necessarily a probability.

Examples:

- Bernoulli model:
$$
  X\sim\operatorname{Bernoulli}(\theta),
$$
  where $\theta=P(X=1)$.

- Normal model with known variance:
$$
  X\sim N(\mu,\sigma_0^2),\qquad \theta=\mu.
$$

- Normal location-scale model:
$$
  X\sim N(\mu,\sigma^2),\qquad
  \theta=(\mu,\sigma^2).
$$

The earlier intuition of $\theta$ as a **"version code"** is useful as long as it is refined: $\theta$ is not an arbitrary label; it represents the model characteristic(s) whose values determine which distribution in the family is being used.

A better wording than “characteristics of the sampled data” is:

> $\theta$ represents characteristics of the **underlying distribution/model**, while sample statistics are used to estimate those characteristics.

Also,

$$
\text{model family}+\theta
\quad\Longrightarrow\quad
\text{specific probability distribution}.
$$

Knowing $\theta$ alone does not tell us the family. For example, $(170,25)$ does not by itself imply a Normal model; the Normal family must already have been assumed.

### 1.2 A target quantity $\psi(\theta)$

Sometimes the full parameter $\theta$ contains more information than we actually care about.

The textbook therefore introduces a function

$$
\psi(\theta),
$$

pronounced **psi of theta**, to represent the particular characteristic we want to estimate.

For example, if

$$
\theta=(\mu,\sigma^2),
$$

then possible targets include

$$
\psi(\theta)=\mu,
\qquad
\psi(\theta)=\sigma,
\qquad
\psi(\theta)=\frac{\mu}{\sigma}.
$$

This gives the general inference pattern

$$
\boxed{
\theta
\longrightarrow
\psi(\theta)
\longleftarrow
T(S)
}
$$

where $T$ is an estimator computed from the sample $S$.

## 2. Population, measurement, sample, and sampling

For a finite population, the textbook uses

$$
\Pi = \{\text{population members}\},
\qquad
N=|\Pi|.
$$

A measurement is a function

$$
X:\Pi\to\mathbb{R}.
$$

For each population member $\pi\in\Pi$, $X(\pi)$ records some measurement.

Examples:

- height;
- fertility score;
- weight;
- a numerical encoding of a category.

A major clarification from the session:

$$
\boxed{N=\text{population size, not the number of all possible events.}}
$$

If we randomly select one population member, then the members of $\Pi$ can also play the role of possible **outcomes** of the sampling experiment. In that induced probability experiment, $\Pi$ behaves like a sample space, but conceptually it is still the population.

An **event** is a set of outcomes, e.g.

$$
\{\pi\in\Pi:X(\pi)\le x\}.
$$

### 2.1 $N$ versus $n$

- $N$: population size;
- $n$: sample size.

A larger $n$ generally gives a more precise estimate because sampling variability decreases; it is **not** because “randomness becomes higher.”

For an i.i.d. sample mean,

$$
\operatorname{Var}(\bar{X})=\frac{\sigma^2}{n}.
$$

So increasing $n$ reduces variability.

However, in **simple random sampling without replacement**, the observations are not exactly independent. If one population member has already been selected, the next draw is made from a slightly changed population.

The important approximation is:

$$
\boxed{
N\text{ large and }\frac{n}{N}\text{ small}
\quad\Rightarrow\quad
\text{sampling without replacement is approximately i.i.d.}
}
$$

Thus two facts coexist:

1. larger $n$ usually improves estimation;
2. the i.i.d. approximation is better when $n/N$ is small.

There is no contradiction: $n$ can be large in absolute terms while still being small relative to $N$.

### 2.2 Simple random sampling and the empirical CDF

The population CDF is

$$
F_X(x)
=
\frac{|\{\pi:X(\pi)\le x\}|}{N}.
$$

From a sample $\pi_1,\ldots,\pi_n$, the empirical CDF is

$$
\hat{F}_X(x)
=
\frac1n\sum_{i=1}^n
I_{(-\infty,x]}(X(\pi_i)).
$$

Under the i.i.d. approximation, the weak law of large numbers gives

$$
\hat{F}_X(x)\xrightarrow{P}F_X(x).
$$

This is an important bridge from the previous material on limits:

$$
\boxed{
\text{LLN}
\Longrightarrow
\text{empirical frequencies become reliable estimates of population probabilities.}
}
$$

### 2.3 Observational studies and selection effects

Random sampling matters because deterministic selection rules can accidentally select a subpopulation.

This creates a **selection effect / selection bias**: the empirical distribution may represent the selected subgroup rather than the target population.

Hence:

- sampling studies are preferred when feasible;
- observational data can still be useful evidence;
- conclusions from observational studies require more care because the data-generating mechanism may not represent the full population.

## 3. Descriptive statistics as first-stage inference

Chapter 5 introduces descriptive statistics as sample-based summaries of unknown population characteristics.

Examples include:

$$
\mu_X
\longrightarrow
\bar{X},
$$

$$
\sigma_X^2
\longrightarrow
S^2,
$$

$$
F_X(x)
\longrightarrow
\hat{F}_X(x),
$$

and population quantiles

$$
x_p=F_X^{-1}(p)
$$

estimated by sample quantiles.

Important special quantiles:

$$
x_{0.5}=\text{median},
\qquad
x_{0.25}=Q_1,
\qquad
x_{0.75}=Q_3.
$$

The interquartile range is

$$
\operatorname{IQR}=Q_3-Q_1.
$$

### 3.1 Mean/SD versus median/IQR

The mean and standard deviation are sensitive to extreme values.

The median and IQR are more robust.

This matters for skewed or heavy-tailed data. In the textbook example, changing a single largest observation from $5$ to $500$ leaves the median unchanged but drastically changes the mean.

That gives the practical rule:

$$
\boxed{
\text{symmetric/light-tailed data: mean and SD often work well}
}
$$

$$
\boxed{
\text{skewed/outlier-prone data: median and IQR can be more stable}
}
$$

## 4. A useful Normal-model decomposition

For the location-scale Normal model, the likelihood contains

$$
\sum_{i=1}^n(x_i-\mu)^2.
$$

The textbook rewrites this as

$$
\boxed{
\sum_{i=1}^n(x_i-\mu)^2
=
n(\bar{x}-\mu)^2
+
\sum_{i=1}^n(x_i-\bar{x})^2.
}
$$

Derivation:

$$
x_i-\mu=(x_i-\bar{x})+(\bar{x}-\mu).
$$

Squaring and summing,

$$
\sum_{i=1}^n(x_i-\mu)^2
=
\sum_{i=1}^n(x_i-\bar{x})^2
+
2(\bar{x}-\mu)\sum_{i=1}^n(x_i-\bar{x})
+
n(\bar{x}-\mu)^2.
$$

But

$$
\sum_{i=1}^n(x_i-\bar{x})=0,
$$

so the cross-term vanishes.

Using

$$
s^2=\frac1{n-1}\sum_{i=1}^n(x_i-\bar{x})^2,
$$

we also get

$$
\sum_{i=1}^n(x_i-\mu)^2
=
n(\bar{x}-\mu)^2+(n-1)s^2.
$$

This identity later explains why the Normal likelihood can be expressed using only $\bar{x}$ and $s^2$.

# Chapter 6 — Likelihood Inference

## 5. Likelihood: fixed data, varying parameter

For observed data $s$, the likelihood is

$$
\boxed{
L(\theta\mid s)=f_\theta(s).
}
$$

The crucial change of viewpoint is:

- in probability, $\theta$ is fixed and the data are random;
- in likelihood inference, the observed data $s$ are fixed and $\theta$ varies.

So likelihood asks:

> Which candidate value of $\theta$ makes the observed data more supported by the model?

A very important distinction:

$$
\boxed{
L(\theta\mid s)\neq P(\theta\mid s).
}
$$

Likelihood is based on the probability/density of the observed data under $\theta$, not on a posterior probability for $\theta$.

### 5.1 Relative likelihood matters

Likelihood inference depends on **relative support**.

For two parameter values,

$$
\frac{L(\theta_1\mid s)}{L(\theta_2\mid s)}
$$

compares how well they are supported by the observed data.

Multiplying the whole likelihood by a positive constant independent of $\theta$ changes nothing:

$$
L^*(\theta\mid s)=cL(\theta\mid s),\qquad c>0.
$$

Therefore likelihoods are considered equivalent up to such positive scaling.

This is why terms that do not depend on $\theta$ are often dropped during simplification.

### 5.2 Continuous data: density is not point probability

For continuous data,

$$
P(X=s)=0.
$$

So $f_\theta(s)$ cannot literally be interpreted as the probability of observing exactly $s$.

Instead, the density determines the probability of sufficiently small neighborhoods around $s$:

$$
P_\theta(a<X<b)
=
\int_a^b f_\theta(x)\,dx.
$$

If one model has larger density near the observed point than another, then sufficiently small neighborhoods around the observation also receive larger probability under that model.

That provides the local interpretation of likelihood for continuous models.

## 6. Sufficient statistics: data reduction without losing likelihood information

A statistic $T(S)$ is sufficient when data values producing the same $T$ value also produce equivalent likelihood functions (up to a positive constant factor).

Conceptually,

$$
\boxed{
\text{sufficiency reduces raw-data detail while preserving information relevant to }\theta.
}
$$

This does **not** mean every detail of the raw observation is preserved.
It means no likelihood information about $\theta$ is lost.

### 6.1 Clarifying the discrete example

In the textbook example,

$$
S=\{1,2,3,4\},
\qquad
\Omega=\{a,b\},
$$

and outcomes $2,3,4$ give the same likelihood ratio between $a$ and $b$.

So a sufficient statistic can map

$$
T(1)=0,
\qquad
T(2)=T(3)=T(4)=1.
$$

This can be viewed as reducing four raw outcome categories to two likelihood-equivalence classes.

However, an important correction from the discussion is:

> This does **not** merge repeated observations into one observation.

If repeated experiments produce

$$
(2,3,4),
$$

those are still three observations, and all three contribute likelihood factors. Sufficiency only says that, for inference about $\theta$, the distinction among the labels $2,3,4$ may be irrelevant **once the total data object is represented through an appropriate statistic**.

### 6.2 Factorization theorem

A standard way to identify sufficient statistics is the factorization theorem:

$$
f_\theta(s)
=
h(s)\,g_\theta(T(s))
\quad\Longrightarrow\quad
T(s)\text{ is sufficient for }\theta.
$$

All dependence on $\theta$ is captured through $T(s)$.

Examples from the Normal model:

- if $\sigma_0^2$ is known and only $\mu$ is unknown, $\bar{X}$ is sufficient;
- if both $\mu$ and $\sigma^2$ are unknown, $(\bar{X},S^2)$ is sufficient.

A **minimal sufficient statistic** achieves the greatest reduction while preserving all likelihood information.

## 7. Maximum likelihood estimation

The maximum likelihood estimator chooses the parameter value that maximizes the likelihood:

$$
\boxed{
\hat\theta_{\mathrm{MLE}}
\in
\arg\max_{\theta\in\Omega}
L(\theta\mid s).
}
$$

Interpretation:

$$
\boxed{
\hat\theta_{\mathrm{MLE}}
=
\text{parameter value best supported by the observed data within the assumed model family.}
}
$$

This does not prove that the model family itself is correct.

### 7.1 Log-likelihood

Because $\log$ is strictly increasing,

$$
\arg\max_\theta L(\theta\mid s)
=
\arg\max_\theta \ell(\theta\mid s),
$$

where

$$
\ell(\theta\mid s)=\log L(\theta\mid s).
$$

For an i.i.d. sample,

$$
L(\theta\mid x_{1:n})
=
\prod_{i=1}^n f_\theta(x_i)
$$

becomes

$$
\boxed{
\ell(\theta\mid x_{1:n})
=
\sum_{i=1}^n \log f_\theta(x_i).
}
$$

The product-to-sum conversion is one of the most important bridges to machine learning: maximizing likelihood is equivalent to minimizing **negative log-likelihood**.

### 7.2 Score function and optimization

For a one-dimensional parameter,

$$
S(\theta\mid s)
=
\frac{\partial \ell(\theta\mid s)}{\partial\theta}.
$$

Candidate stationary points solve

$$
S(\theta\mid s)=0.
$$

A negative second derivative,

$$
\frac{\partial^2\ell}{\partial\theta^2}<0,
$$

indicates local downward curvature and therefore a local maximum.

But solving the score equation is not enough by itself:

- the point may be a minimum;
- it may only be a local maximum;
- the global maximum may lie on a boundary.

For a multidimensional parameter

$$
\theta=(\theta_1,\ldots,\theta_k),
$$

the score becomes the gradient

$$
\nabla_\theta \ell(\theta\mid s),
$$

and second-order curvature is described by the Hessian.

### 7.3 Normal location-scale MLE

For

$$
X_i\overset{\mathrm{iid}}{\sim}N(\mu,\sigma^2),
$$

the MLE is

$$
\boxed{
\hat\mu_{\mathrm{MLE}}=\bar{X}
}
$$

and

$$
\boxed{
\hat\sigma^2_{\mathrm{MLE}}
=
\frac1n\sum_{i=1}^n(X_i-\bar{X})^2.
}
$$

Compare this with the usual unbiased sample variance,

$$
S^2
=
\frac1{n-1}\sum_{i=1}^n(X_i-\bar{X})^2.
$$

Thus,

$$
\hat\sigma^2_{\mathrm{MLE}}
=
\frac{n-1}{n}S^2.
$$

This is a useful example showing that **MLE and unbiased estimation are different criteria**.

### 7.4 Small numerical sanity check: Bernoulli MLE

In [1]:
import numpy as np

n = 10
heads = 7
theta_grid = np.linspace(0.001, 0.999, 999)

log_likelihood = heads*np.log(theta_grid) + (n-heads)*np.log(1-theta_grid)
theta_hat = theta_grid[np.argmax(log_likelihood)]

print(f"Grid MLE ≈ {theta_hat:.3f}")
print(f"Analytic MLE = heads/n = {heads/n:.3f}")

Grid MLE ≈ 0.700
Analytic MLE = heads/n = 0.700


For Bernoulli data with 7 heads in 10 trials,

$$
L(\theta)\propto \theta^7(1-\theta)^3,
$$

and

$$
\hat\theta_{\mathrm{MLE}}=\frac7{10}=0.7.
$$

This is the simplest example of likelihood estimation becoming an optimization problem.

## 8. Estimators, estimates, and notation

A recurring source of confusion was distinguishing the random estimator from its observed numerical value.

For example:

$$
\bar{X}
=
\frac1n\sum_{i=1}^n X_i
$$

is a random variable before the data are observed, while

$$
\bar{x}
=
\frac1n\sum_{i=1}^n x_i
$$

is the realized value after observing the sample.

Similarly,

$$
S^2
=
\frac1{n-1}\sum_{i=1}^n(X_i-\bar{X})^2
$$

is the sample-variance estimator, while

$$
s^2
=
\frac1{n-1}\sum_{i=1}^n(x_i-\bar{x})^2
$$

is the numerical sample variance from one dataset.

The underlying population parameter is $\sigma^2$.

So:

$$
\boxed{
\sigma^2
\quad\leftarrow\quad
S^2
\quad\xrightarrow{\text{observe data}}\quad
s^2.
}
$$

## 9. $E_\theta$, bias, variance, and MSE

The notation

$$
E_\theta[\cdot]
$$

means expectation under the distribution $P_\theta$.

For a discrete model,

$$
E_\theta[X]
=
\sum_x x\,p_\theta(x),
$$

and for a continuous model,

$$
E_\theta[X]
=
\int x f_\theta(x)\,dx.
$$

The subscript $\theta$ matters because changing $\theta$ generally changes the distribution and therefore the expectation.

### 9.1 Mean-squared error

For an estimator $T$ of $\psi(\theta)$,

$$
\boxed{
\operatorname{MSE}_\theta(T)
=
E_\theta[(T-\psi(\theta))^2].
}
$$

The key decomposition is

$$
\boxed{
\operatorname{MSE}_\theta(T)
=
\operatorname{Var}_\theta(T)
+
\left(E_\theta[T]-\psi(\theta)\right)^2.
}
$$

The second term is the squared bias:

$$
\operatorname{Bias}_\theta(T)
=
E_\theta[T]-\psi(\theta).
$$

So

$$
\boxed{
\text{MSE}
=
\text{variance}
+
\text{bias}^2.
}
$$

Interpretation:

- **variance**: sample-to-sample fluctuation of the estimator;
- **bias**: displacement of the estimator's center from the true target.

### 9.2 Why can the bias term leave the expectation?

In the proof, the term

$$
E_\theta\left[
(E_\theta(T)-\psi(\theta))^2
\right]
$$

simplifies to

$$
(E_\theta(T)-\psi(\theta))^2
$$

because, for fixed $\theta$,

$$
E_\theta(T)-\psi(\theta)
$$

is just a constant.

The same fact makes the cross-term vanish:

$$
E_\theta[T-E_\theta(T)]=0.
$$

This is the algebraic reason behind the bias-variance decomposition.

## 10. Standard deviation versus standard error

This distinction became one of the clearest conceptual takeaways of the session.

### Standard deviation

For a random variable $X$,

$$
\operatorname{SD}(X)
=
\sqrt{\operatorname{Var}(X)}.
$$

It measures the spread of **individual observations**.

### Standard error

For an estimator $T$,

$$
\boxed{
\operatorname{SE}(T)
=
\sqrt{\operatorname{Var}(T)}.
}
$$

It is the standard deviation of the estimator's **sampling distribution**.

Thus:

$$
\boxed{
\text{SD: spread of data}
\qquad
\text{SE: spread of an estimator across repeated samples}.
}
$$

For the sample mean,

$$
\operatorname{SE}(\bar{X})
=
\frac{\sigma}{\sqrt n}.
$$

When $\sigma$ is unknown, it is commonly estimated by

$$
\widehat{\operatorname{SE}}(\bar{X})
=
\frac{s}{\sqrt n}.
$$

The term “standard error” is not restricted to means. Any estimator can have a standard error.

## 11. Consistency

An estimator sequence $T_n$ is **consistent in probability** for $\psi(\theta)$ if

$$
T_n\xrightarrow{P_\theta}\psi(\theta)
\qquad
\text{as }n\to\infty.
$$

It is **almost surely consistent** if

$$
T_n\xrightarrow{\mathrm{a.s.}}\psi(\theta).
$$

The intuition is simple:

$$
\boxed{
\text{as the amount of data increases, a sensible estimator should converge toward the truth.}
}
$$

This connects statistical inference directly back to the laws of large numbers.

## 12. Confidence intervals

A $\gamma$-confidence interval is a random interval

$$
C(S)=(l(S),u(S))
$$

constructed so that

$$
P_\theta\bigl(\psi(\theta)\in C(S)\bigr)\ge\gamma.
$$

The important frequentist interpretation is about the **procedure under repeated sampling**.

After one specific dataset is observed, the resulting interval either contains the fixed true parameter or it does not.

Therefore it is generally not correct to say:

> “This realized 95% confidence interval has 95% probability of containing the fixed parameter.”

The 95% property is a long-run coverage property of the interval-producing procedure.

### 12.1 z-confidence interval for a Normal mean

If

$$
X_i\sim N(\mu,\sigma_0^2)
$$

with known $\sigma_0^2$, then

$$
Z
=
\frac{\bar{X}-\mu}{\sigma_0/\sqrt n}
\sim N(0,1).
$$

A $\gamma$-confidence interval is

$$
\boxed{
\bar{X}
\pm
z_{(1+\gamma)/2}
\frac{\sigma_0}{\sqrt n}.
}
$$

For $\gamma=0.95$,

$$
z_{0.975}\approx1.96,
$$

so

$$
\boxed{
\bar{X}
\pm
1.96\frac{\sigma_0}{\sqrt n}.
}
$$

The half-width,

$$
z_{(1+\gamma)/2}\operatorname{SE}(\bar{X}),
$$

is the **margin of error**.

### 12.2 Confidence intervals are not always symmetric

For the Normal variance,

$$
\frac{(n-1)S^2}{\sigma^2}
\sim
\chi^2_{n-1},
$$

which gives an exact confidence interval of the form

$$
\left[
\frac{(n-1)s^2}{\chi^2_{(1+\gamma)/2}(n-1)},
\;
\frac{(n-1)s^2}{\chi^2_{(1-\gamma)/2}(n-1)}
\right].
$$

This interval is generally asymmetric.

So the familiar pattern

$$
\hat\theta\pm c\,\operatorname{SE}(\hat\theta)
$$

is useful, but it is not the universal form of every confidence interval.

## 13. Hypothesis testing and P-values

A null hypothesis has the form

$$
H_0:\psi(\theta)=\psi_0.
$$

The basic question is:

> If $H_0$ were true, how unusual would the observed data be?

The P-value is a probability measuring this extremeness or **surprise under the null**.

A small P-value means that data at least as extreme as what we observed would be unlikely if $H_0$ were true.

Two crucial corrections:

$$
\boxed{
\text{P-value}\neq P(H_0\mid\text{data})
}
$$

and

$$
\boxed{
\text{a large P-value does not prove }H_0\text{ is true.}
}
$$

### 13.1 z-test for a Normal mean

Suppose

$$
X_i\sim N(\mu,\sigma_0^2)
$$

with known $\sigma_0^2$, and we test

$$
H_0:\mu=\mu_0.
$$

Under $H_0$,

$$
Z
=
\frac{\bar{X}-\mu_0}{\sigma_0/\sqrt n}
\sim N(0,1).
$$

For a two-sided test, the P-value is

$$
\boxed{
p
=
2\left[
1-\Phi\left(
\left|
\frac{\bar{x}-\mu_0}{\sigma_0/\sqrt n}
\right|
\right)
\right].
}
$$

The factor 2 appears because deviations in **both directions** count as equally extreme.

### 13.2 What “surprising” means

“Surprising” is not informal language here.

It means:

$$
\boxed{
\text{the observed statistic lies in a region of low probability under }H_0.
}
$$

For the textbook figure with

$$
\mu_0=3,
\qquad
\sigma_0^2=1,
\qquad
n=10,
\qquad
\bar{x}=4.2,
$$

the sampling distribution under $H_0$ is

$$
\bar{X}\sim N\left(3,\frac1{10}\right).
$$

The observed value $4.2$ lies far in the right tail.

The two-sided P-value also counts equally extreme values in the left tail.

### 13.3 Density of the MLE is not the P-value

In that same figure, the Normal curve is the **sampling density of the MLE**

$$
\hat\mu=\bar{X}
$$

under $H_0$.

It is not a plot of the P-value.

The P-value corresponds to the total **tail area** at least as extreme as the observed statistic.

Also, a density can exceed 1.

For a continuous variable,

$$
\text{density height}\neq\text{probability}.
$$

Probability is area:

$$
P(a\le X\le b)
=
\int_a^b f_X(x)\,dx.
$$

The PDF may be taller than 1 if it is sufficiently narrow; only the total area must equal 1.

## 14. Statistical significance versus practical significance

With large enough $n$, even a very small difference between the true parameter and the null value can become statistically detectable.

For the z-statistic,

$$
\frac{|\bar{X}-\mu_0|}{\sigma_0/\sqrt n},
$$

the denominator decreases as $n$ increases.

Therefore:

$$
\boxed{
\text{statistically significant}
\not\Rightarrow
\text{practically important}.
}
$$

Practical significance concerns the magnitude of the effect in the application, not merely whether the P-value is small.

## 15. Power and sample-size design

Instead of only asking what uncertainty remains after collecting data, we can reverse the problem:

$$
\boxed{
\text{How much data should we collect to achieve a desired accuracy or sensitivity?}
}
$$

For a test with significance threshold $\alpha$, the **power function** is

$$
\beta(\theta)
=
P_\theta(\text{P-value}<\alpha).
$$

Power measures the ability of the procedure to detect departures from the null.

This is one reason a large P-value should not be interpreted as evidence that the null is true: the test may simply have low power against the relevant alternatives.

## 16. Distribution-free methods

Likelihood inference assumes that the true distribution belongs to a specified family

$$
\{P_\theta:\theta\in\Omega\}.
$$

If the model family is wrong, likelihood-based inferences can be misleading.

Distribution-free methods weaken the modeling assumptions by allowing a much larger class of possible distributions.

This creates a trade-off:

$$
\boxed{
\text{stronger assumptions}
\Rightarrow
\text{potentially more efficient inference if correct}
}
$$

versus

$$
\boxed{
\text{weaker assumptions}
\Rightarrow
\text{greater robustness to misspecification}.
}
$$

### 16.1 Method of moments

For population moments

$$
\mu_i=E(X^i),
$$

the corresponding sample moments are

$$
m_i
=
\frac1n\sum_{j=1}^n X_j^i.
$$

The method-of-moments principle estimates

$$
\psi(\mu_1,\ldots,\mu_k)
$$

by

$$
\psi(m_1,\ldots,m_k).
$$

The textbook also introduces the delta theorem as a way to transfer asymptotic Normal behavior through smooth transformations.

For the current learning plan, the main idea is more important than the full derivation:

$$
\boxed{
\text{sample moments approximate population moments, and smooth functions of them can inherit asymptotic approximations.}
}
$$

A practical warning: high-order moments are sensitive to extreme values and can require very large sample sizes.

### 16.2 Bootstrapping

The empirical distribution is

$$
\hat{F}(x)
=
\frac1n
\sum_{i=1}^n I_{(-\infty,x]}(x_i).
$$

If the observations are distinct, $\hat{F}$ places probability $1/n$ on each observed value.

Therefore, sampling from $\hat{F}$ is equivalent to:

$$
\boxed{
\text{sampling from the observed data with replacement}.
}
$$

The bootstrap procedure is:

1. treat the empirical distribution $\hat{F}$ as an estimate of the unknown population distribution;
2. draw many resamples of size $n$ with replacement;
3. compute the estimator on each resample;
4. use the distribution of those bootstrap estimates to approximate sampling variability, bias, standard error, or other properties.

Conceptually,

$$
\boxed{
\text{bootstrap}
=
\text{empirical distribution}
+
\text{Monte Carlo resampling}.
}
$$

This is especially useful when the estimator's analytical sampling distribution is difficult to derive, e.g. for a sample median.

## 17. Main learning checkpoints and corrections

| Initial idea / confusion | Refined understanding |
|---|---|
| $N$ is the number of all possible events | $N$ is the population size. Under random selection, population members can act as possible outcomes, while events are sets of outcomes. |
| Larger $n$ is better because randomness is higher | Larger $n$ usually reduces estimator variability. For sampling without replacement, approximate i.i.d. behavior depends on $n/N$ being small. |
| $\theta$ is sometimes a probability, sometimes just a code | $\theta$ indexes the model family and represents the unknown model characteristic(s). Sometimes that characteristic is itself a probability. |
| A sufficient statistic merges multiple observations into one | Sufficiency removes distinctions in the raw data that carry no additional likelihood information about $\theta$; repeated observations still all contribute. |
| $E_\theta(T)$ still behaves like a random quantity inside the MSE proof | For fixed $\theta$, $E_\theta(T)$ is a constant. |
| $S=X_1+\cdots+X_n$ | In this context, $S^2$ is sample variance and $S$ is sample standard deviation. |
| SD and SE are almost the same thing | SD measures spread of observations; SE measures spread of an estimator's sampling distribution. |
| A small P-value means the null is unlikely to be true | A P-value is $P(\text{data at least this extreme}\mid H_0)$, not $P(H_0\mid\text{data})$. |
| A high PDF value over 1 is impossible | Continuous density can exceed 1; only integrated probability must remain between 0 and 1. |

## 18. Connections to machine learning

Several ideas from Chapters 5–6 transfer directly into machine learning.

### Statistical model and parameters

Machine-learning models are also parameterized families:

$$
p_\theta(y\mid x),
\qquad
f_\theta(x),
\qquad
\theta\in\mathbb{R}^d.
$$

In deep learning, $\theta$ may contain millions or billions of weights.

### MLE as optimization

For i.i.d. data,

$$
\hat\theta_{\mathrm{MLE}}
=
\arg\max_\theta
\sum_{i=1}^n \log p_\theta(x_i).
$$

Equivalently,

$$
\hat\theta_{\mathrm{MLE}}
=
\arg\min_\theta
\left[
-\sum_{i=1}^n \log p_\theta(x_i)
\right].
$$

This is the origin of many familiar loss functions.

### Gradient-based learning

The score

$$
\nabla_\theta\ell(\theta)
$$

is conceptually the same object that appears in gradient-based parameter optimization.

### Bias, variance, and uncertainty

The decomposition

$$
\operatorname{MSE}
=
\operatorname{Var}
+
\operatorname{Bias}^2
$$

is an early version of the bias-variance reasoning that becomes central in statistical learning.

### Bootstrap and resampling

Bootstrap-style resampling appears in uncertainty estimation, model evaluation, bagging, and ensemble methods.

## 19. Scope choices

This session deliberately did **not** try to master every advanced statistical detail.

Lower-priority or lightly treated material includes:

- exact implementation details of every sample-quantile convention;
- full technical proofs of the delta theorem;
- deeper numerical optimization theory for multidimensional MLEs;
- detailed distribution-free sign-statistic methods;
- advanced asymptotic theory in later parts of Chapter 6;
- full sample-size/power derivations beyond the main ideas.

The priority was to understand the concepts that form the statistical foundation for machine learning.

## 20. Follow-up work

### Main conceptual follow-ups

- Bayesian inference: compare likelihood-based inference with posterior inference.
- Model checking: understand what happens when the assumed family $\{P_\theta\}$ is inappropriate.
- Large-sample behavior of MLEs: revisit when asymptotic normality becomes useful later.
- Deeper power/type-I/type-II error analysis if needed.

### Separate computational notebook

The textbook includes computer exercises for these chapters. These should be implemented in a **separate practice notebook** . The current notebook remains focused on the conceptual learning process.

Good candidates for that implementation notebook:

- plot Bernoulli/Binomial likelihoods and log-likelihoods;
- numerically recover MLEs;
- compare $1/n$ and $1/(n-1)$ variance estimators;
- simulate estimator bias/variance/MSE;
- visualize standard error shrinking with $n$;
- simulate confidence-interval coverage;
- simulate P-values and test power;
- bootstrap means and medians.

## 21. Discussions

- Session discussion and derivations, especially:
  - population size $N$ versus sample size $n$;
  - derivation of
$$
    \sum_i(x_i-\mu)^2
    =
    n(\bar{x}-\mu)^2
    +
    \sum_i(x_i-\bar{x})^2;
$$
  - meaning of $\theta$ and $\psi(\theta)$;
  - sufficient statistics and repeated observations;
  - bias-variance decomposition;
  - standard deviation versus standard error;
  - z-tests, P-values, and the interpretation of “surprising”;
  - density of an estimator versus a P-value;
  - distribution-free inference and bootstrapping.

---

### Session takeaway

The major change in perspective is:

$$
\boxed{
\text{Probability: model }\rightarrow\text{ data}
}
$$

versus

$$
\boxed{
\text{Statistics: data }\rightarrow\text{ model/parameter}.
}
$$

Likelihood then turns inference into a relative-support problem over parameters, and MLE turns that into an optimization problem. The rest of the chapter asks how reliable that estimate is and how to quantify uncertainty around it.

That chain is the key statistical foundation to carry forward into machine learning.